# MPA-MLF Final Project — Room Occupancy Classification
**Author:** Rafael Estévez Marrero  
**Task:** Classify number of persons (0, 1, 2, 3) in a room from 60 GHz delay-Doppler snapshots  
**Dataset:** PNG images (delay-Doppler domain)  

---
## 0. Setup & Drive Mount

In [ ]:
# Mount Google Drive (datasets must be uploaded there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CONFIGURE PATHS ──────────────────────────────────────────────────────────
# Change BASE_DIR to wherever you uploaded the zips in your Drive
BASE_DIR    = '/content/drive/MyDrive/MPA_MLF_Final'

TRAIN_ZIP   = f'{BASE_DIR}/x_train.zip'
TEST_ZIP    = f'{BASE_DIR}/x_test.zip'
Y_TRAIN_CSV = f'{BASE_DIR}/y_train_v2.csv'
SUBM_CSV    = f'{BASE_DIR}/y_test_submission_example_v2.csv'

TRAIN_DIR   = '/content/x_train'
TEST_DIR    = '/content/x_test'
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import zipfile, os

if not os.path.exists(TRAIN_DIR):
    print('Extracting training data...')
    with zipfile.ZipFile(TRAIN_ZIP, 'r') as z:
        z.extractall(TRAIN_DIR)
    print('Done.')

if not os.path.exists(TEST_DIR):
    print('Extracting test data...')
    with zipfile.ZipFile(TEST_ZIP, 'r') as z:
        z.extractall(TEST_DIR)
    print('Done.')

# List a few files to verify
train_files = sorted(os.listdir(TRAIN_DIR))
test_files  = sorted(os.listdir(TEST_DIR))
print(f'Train images: {len(train_files)}, Test images: {len(test_files)}')
print('Sample train:', train_files[:5])
print('Sample test: ', test_files[:5])

## 1. Imports & Reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import (confusion_matrix, classification_report,
                              f1_score, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## 2. Exploratory Data Analysis

In [ ]:
y_train_df = pd.read_csv(Y_TRAIN_CSV)
print(y_train_df.head(10))
print(f'\nTotal training samples: {len(y_train_df)}')
print('\nClass distribution:')
print(y_train_df['target'].value_counts().sort_index())

In [ ]:
# Plot class distribution
fig, ax = plt.subplots(figsize=(7, 4))
counts = y_train_df['target'].value_counts().sort_index()
ax.bar(['0 – Machine\n(no person)', '1 – One\nperson',
        '2 – Two\npersons', '3 – Three\npersons'],
       counts.values, color=['steelblue','orange','green','red'])
ax.set_title('Class Distribution in Training Set', fontsize=13)
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 20, str(v), ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Visualise one sample per class
# NOTE: images are numbered from 1; labels (y_train) are indexed from 0
# → image index = label row index + 1

CLASS_NAMES = ['Machine only (0 persons)', '1 person', '2 persons', '3 persons']
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

for cls in range(4):
    idx = y_train_df[y_train_df['target'] == cls].iloc[0]['id']  # label id (0-based)
    img_path = os.path.join(TRAIN_DIR, f'{int(idx)+1}.png')       # image numbering from 1
    img = Image.open(img_path).convert('RGB')
    axes[cls].imshow(np.array(img))
    axes[cls].set_title(CLASS_NAMES[cls], fontsize=10)
    axes[cls].axis('off')

plt.suptitle('Sample delay-Doppler snapshots per class', fontsize=12)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150)
plt.show()

## 3. Dataset & Data Augmentation

In [ ]:
IMG_SIZE = 64   # resize to 64×64 (images are small delay-Doppler maps)

# ── TRAINING TRANSFORMS (with augmentation) ──────────────────────────────────
train_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.3),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# ── VALIDATION / TEST TRANSFORMS (no augmentation) ───────────────────────────
val_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

In [ ]:
class DelayDopplerDataset(Dataset):
    """
    Dataset for delay-Doppler PNG images.
    img_dir  : folder containing the PNG files
    labels_df: DataFrame with columns ['id', 'target']
               id is 0-based → filename = id+1
    transform: torchvision transforms
    is_test  : if True, no labels; ids are taken from the submission example
    """
    def __init__(self, img_dir, labels_df=None, transform=None, is_test=False):
        self.img_dir   = img_dir
        self.transform = transform
        self.is_test   = is_test

        if not is_test:
            self.ids    = labels_df['id'].values
            self.labels = labels_df['target'].values
        else:
            # Test images: filenames in sorted order
            fnames = sorted(os.listdir(img_dir),
                            key=lambda f: int(f.replace('.png','')))
            self.fnames = fnames
            self.img_ids = [int(f.replace('.png','')) for f in fnames]

    def __len__(self):
        return len(self.ids) if not self.is_test else len(self.fnames)

    def __getitem__(self, idx):
        if not self.is_test:
            img_name = f'{int(self.ids[idx])+1}.png'   # label id → image filename
            label    = int(self.labels[idx])
        else:
            img_name = self.fnames[idx]
            label    = -1  # unknown

        img_path = os.path.join(self.img_dir, img_name)
        img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, label

In [ ]:
# ── Train / Validation split (80 / 20, stratified) ───────────────────────────
train_df, val_df = train_test_split(
    y_train_df, test_size=0.2, random_state=SEED,
    stratify=y_train_df['target']
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f'Training samples  : {len(train_df)}')
print(f'Validation samples: {len(val_df)}')

train_dataset = DelayDopplerDataset(TRAIN_DIR, train_df, transform=train_transforms)
val_dataset   = DelayDopplerDataset(TRAIN_DIR, val_df,   transform=val_transforms)
test_dataset  = DelayDopplerDataset(TEST_DIR,  transform=val_transforms, is_test=True)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'\nBatches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}')

## 4. Model — EfficientNet-B0 (Transfer Learning)

In [ ]:
def build_model(num_classes=4, freeze_backbone=False):
    """
    EfficientNet-B0 with a custom classification head.
    BatchNorm + Dropout for regularisation.
    """
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes)
    )
    return model


model = build_model(num_classes=4, freeze_backbone=False).to(DEVICE)
print(model.classifier)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {total_params:,}')

## 5. Training Setup

In [ ]:
# ── Class weights to handle class imbalance ───────────────────────────────────
from torch.utils.data import WeightedRandomSampler
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    'balanced', classes=np.array([0,1,2,3]),
    y=train_df['target'].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print('Class weights:', class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# AdamW with weight decay
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Cosine annealing LR scheduler
EPOCHS = 40
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

## 6. Training Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
CKPT_PATH = '/content/best_model.pth'

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), CKPT_PATH)
        star = ' ★ saved'
    else:
        star = ''

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch [{epoch:02d}/{EPOCHS}]  '
              f'Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f}  |  '
              f'Val Loss: {va_loss:.4f}  Acc: {va_acc:.4f}{star}')

print(f'\nBest validation accuracy: {best_val_acc:.4f}')

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Validation')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'],   label='Validation')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 8. Evaluation on Validation Set

In [ ]:
# Load best model
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))

_, val_acc, val_preds, val_labels = evaluate(model, val_loader, criterion, DEVICE)

print(f'Validation Accuracy: {val_acc:.4f}')
print(f'\nMacro F1-score: {f1_score(val_labels, val_preds, average="macro"):.4f}')
print(f'Weighted F1-score: {f1_score(val_labels, val_preds, average="weighted"):.4f}')

print('\nDetailed Classification Report:')
print(classification_report(val_labels, val_preds,
                             target_names=CLASS_NAMES))

In [ ]:
# Confusion matrix
cm = confusion_matrix(val_labels, val_preds)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['0-Machine', '1-Person',
                                               '2-Persons', '3-Persons'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Validation Set', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Per-class F1 bar chart
f1_per_class = f1_score(val_labels, val_preds, average=None)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['0-Machine','1-Person','2-Persons','3-Persons'],
       f1_per_class,
       color=['steelblue','orange','green','red'])
ax.set_ylim(0, 1)
ax.set_ylabel('F1-score')
ax.set_title('Per-class F1-score (Validation Set)', fontsize=12)
for i, v in enumerate(f1_per_class):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.savefig('f1_per_class.png', dpi=150)
plt.show()

## 9. Fine-Tuning Run (optional second stage)

If accuracy is not satisfying, run this cell to do a second fine-tuning pass with a lower learning rate and all layers unfrozen.

In [ ]:
# Unfreeze all layers and fine-tune with a lower LR
for param in model.parameters():
    param.requires_grad = True

optimizer_ft = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=20, eta_min=1e-7)

FINETUNE_EPOCHS = 20
for epoch in range(1, FINETUNE_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer_ft, criterion, DEVICE)
    va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    scheduler_ft.step()

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), CKPT_PATH)
        star = ' ★ saved'
    else:
        star = ''

    if epoch % 5 == 0 or epoch == 1:
        print(f'FT Epoch [{epoch:02d}/{FINETUNE_EPOCHS}]  '
              f'Train: {tr_acc:.4f}  Val: {va_acc:.4f}{star}')

print(f'\nBest validation accuracy after fine-tuning: {best_val_acc:.4f}')

## 10. Test Set Predictions & Kaggle Submission

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

all_preds = []
with torch.no_grad():
    for imgs, _ in tqdm(test_loader, desc='Predicting test set'):
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())

print(f'Total test predictions: {len(all_preds)}')

In [ ]:
# Build submission DataFrame matching the required format
subm_example = pd.read_csv(SUBM_CSV)

# Test image IDs (filename numbers → label id = filename_number - 1)
test_img_ids = test_dataset.img_ids          # list of filename numbers, e.g. [9228, 9229, ...]
test_label_ids = [x - 1 for x in test_img_ids]   # convert to label id

submission_df = pd.DataFrame({'id': test_label_ids, 'target': all_preds})
submission_df = submission_df.sort_values('id').reset_index(drop=True)

print(submission_df.head(10))
print(f'\nPrediction distribution:')
print(submission_df['target'].value_counts().sort_index())

SUBMISSION_PATH = '/content/submission.csv'
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f'\nSaved to: {SUBMISSION_PATH}')

In [ ]:
# Download submission file
from google.colab import files
files.download(SUBMISSION_PATH)

## 11. Summary

| Metric | Value |
|--------|-------|
| Architecture | EfficientNet-B0 (ImageNet pretrained) |
| Input size | 64×64 RGB |
| Data augmentation | HFlip, VFlip, Rotation±15°, ColorJitter, GaussianBlur |
| Regularisation | Dropout (0.4+0.3), Weight decay (1e-4), BatchNorm1d |
| Optimiser | AdamW, lr=1e-3 → CosineAnnealing |
| Class imbalance | CrossEntropyLoss with class weights |
| Epochs (total) | 40 + 20 fine-tune |
| Best Val Accuracy | *see above* |
| Macro F1 | *see above* |